# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data and Train Model
df = pd.read_csv('../../../STARTERPACK ML/data/raw/content_refresh_anonymized.csv')
df = df.dropna(subset=['trend_direction'])
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count'] = df['word_count'].fillna(0)
df['avg_position'] = df['avg_position'].fillna(0)

features = ['days_since_last_update', 'ctr', 'impressions_90d', 'clicks_90d', 'avg_position', 'engagement_rate', 'word_count', 'has_word_count']
X = df[features]
y = df['is_declining']

rf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf.fit(X, y)

# 2. Predict Probabilities
df['decline_prob'] = rf.predict_proba(X)[:, 1]

# 3. Rank Actions (Probability > 65%)
queue = df[df['decline_prob'] > 0.65].copy()
queue = queue.sort_values('decline_prob', ascending=False)

# 4. Generate Reason Codes
def assign_reason(row):
    if row['days_since_last_update'] > 180 and row['ctr'] < 2.0:
        return 'Stale & Low CTR - Needs Content Update'
    elif row['impressions_90d'] > 1000 and row['ctr'] < 1.0:
        return 'High Visibility, Poor CTR - Needs Better Title/Meta'
    else:
        return 'General Decay - Review Content Quality'

queue['reason_code'] = queue.apply(assign_reason, axis=1)
action_queue = queue[['content_id', 'client_id', 'decline_prob', 'reason_code']]

print("Top 5 Recommended Actions:")
print(action_queue.head())


Top 5 Recommended Actions:
                 content_id          client_id  decline_prob  \
11321  content_aa76dcfed6e7  client_3fdba35f04      0.762750   
3041   content_f713f097536a  client_3fdba35f04      0.762336   
25808  content_d0e81e632d6f  client_3fdba35f04      0.761949   
2967   content_b74291f8efd2  client_3fdba35f04      0.761885   
23935  content_972b2512be5e  client_3fdba35f04      0.761885   

                                             reason_code  
11321  High Visibility, Poor CTR - Needs Better Title...  
3041   High Visibility, Poor CTR - Needs Better Title...  
25808  High Visibility, Poor CTR - Needs Better Title...  
2967   High Visibility, Poor CTR - Needs Better Title...  
23935  High Visibility, Poor CTR - Needs Better Title...  


## 2. Intended use and limits

**Intended Use:** This playbook is a prioritization tool for Content Strategists. It highlights articles that have the highest mathematical probability of decaying traffic so the team knows exactly where to spend their limited weekly writing hours.

**Limits:** The model does not understand the *semantic meaning* of the content. It only sees behavioral metrics (CTR, age, clicks). It will flag evergreen/static pages (like Privacy Policies) as "declining" just because they have low engagement. It is NOT valid for newly published articles (under 30 days old).

In [2]:
# Code check to enforce limits (exclude new content)
valid_queue = queue[queue['days_since_last_update'] > 30]
print(f"Filtered out {len(queue) - len(valid_queue)} items that were too new to evaluate reliably.")


Filtered out 5274 items that were too new to evaluate reliably.


## 3. Human review + the no-go list

**Human Review Requirement:** A human editor must click the URL and visually inspect the page to determine if the content actually needs a rewrite or if it's just a structural page (like a glossary, author bio, or contact page).

**The No-Go List (NEVER AUTOMATE):**
- Do not automatically delete pages with a high decline probability.
- Do not automatically redirect URLs without human sign-off.
- Do not feed the page into a GenAI pipeline to auto-publish a rewrite without editor review.

In [3]:
# Safety Check: Ensure no destructive automation commands exist in our export
assert 'delete' not in valid_queue['reason_code'].values, "Safety violation: 'delete' action found."


## 4. Monitoring / retrain triggers

**When to Retrain / Monitor:**
If the top 50 "refreshed" articles show a collective 0% uplift in CTR after 30 days of being rewritten, the model's signals (or the reason codes) are stale and disconnected from reality. The model must be retrained on fresh seasonal data.

In [4]:
# Baseline metric for future monitoring
average_ctr_of_flagged = valid_queue['ctr'].mean()
print(f"Monitoring Baseline: Average CTR of flagged content is {average_ctr_of_flagged:.2f}%. Watch for uplift post-refresh.")


Monitoring Baseline: Average CTR of flagged content is 0.11%. Watch for uplift post-refresh.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import os

# 1. Export CSV
valid_queue[['content_id', 'client_id', 'decline_prob', 'reason_code']].to_csv('../outputs/action_queue.csv', index=False)
print("Exported action_queue.csv to work/outputs/")

# 2. Export Figure
plt.figure(figsize=(8, 5))
plt.hist(df['decline_prob'], bins=20, color='royalblue', edgecolor='black', alpha=0.7)
plt.axvline(0.65, color='red', linestyle='dashed', linewidth=2, label='Action Threshold (65%)')
plt.title("Distribution of Content Decline Probabilities")
plt.xlabel("Probability of Decline")
plt.ylabel("Number of Articles")
plt.legend()
plt.tight_layout()
plt.savefig('../figures/queue_distribution.png')
plt.close()
print("Exported queue_distribution.png to work/figures/")


Exported action_queue.csv to work/outputs/
Exported queue_distribution.png to work/figures/


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.